### Import Dependencies

In [11]:
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
import os

from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.agents.run_config import RunConfig

from google.genai import types

from dotenv import load_dotenv

load_dotenv("../../.env")

from utils.tools_2 import check_warehouse_availability, reserve_warehouse_items

### ADK Agent

In [12]:
model = LiteLlm(
    model="openai/gpt-5.4-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
)

In [13]:
warehouse_agent = Agent(
    name="warehouse_manager_agent",
    model=model,
    tools=[
        check_warehouse_availability,
        reserve_warehouse_items
    ],
    description="The user is asking to reserve items from the warehouses or about availability of the items in warehouses.",
    instruction="""You are a part of the shopping assistant that can manage available inventory in the warehouses.

## Instructions

- As the final answer you should return an answer to the users query in a form of actions performed.
- You must always check the availability of the items in the warehouses before reserving them.
- Only reserve items in warehouses if entire order can be reserved or the user has confirmed that they want a partial reservation.
- If you cannot reserve any items, return an answer that the order cannot be reserved.
- If you can reserve some items, return an answer that the order can be partially reserved and include the details.
- If only partial quantity can be reserved in some warehouses, try to combine the required quantity from different warehouses.
- Try to reserve items from the closest warehouse to the user first if users location is provided.
- As the final answer you should return an answer in a form of actions performed.
"""
)


### ADK Session

In [14]:
session_service = InMemorySessionService()

In [15]:
await session_service.create_session(
    app_name="warehouse_app",
    user_id="user_1",
    session_id="sessions_1"
)

Session(id='sessions_1', app_name='warehouse_app', user_id='user_1', state={}, events=[], last_update_time=1789464203.362111)

In [16]:
runner = Runner(
    agent=warehouse_agent,
    session_service=session_service,
    app_name="warehouse_app"
)

In [17]:
message = types.Content(
    role="user",
    parts=[
        types.Part(
            text="What is the availability of B09X1LDMH6 in all of your warehouses?"
        )
    ]
)

In [18]:
result = runner.run(
    user_id="user_1",
    session_id="sessions_1",
    new_message=message,
    run_config=RunConfig(
        max_llm_calls=5
    )
)  

In [19]:
result

<generator object Runner.run at 0x11f244320>

In [20]:
for event in result:
    print("======================")
    print(event)

model_version='gpt-5.4-mini' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'items': [
            {<... 2 items at Max depth ...>},
          ]
        },
        id='call_RlTFsxhel0tzLUA3u1bmPzkr',
        name='check_warehouse_availability'
      )
    ),
  ],
  role='model'
) grounding_metadata=None partial=False turn_complete=None turn_complete_reason=None interaction_status=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  cached_content_token_count=0,
  candidates_token_count=37,
  prompt_token_count=561,
  total_token_count=598
) live_session_resumption_update=None live_session_id=None go_away=None voice_activity=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None interaction_id=None environment_id=None invocation_id='e-4b

### Agent Run Wrapper

In [21]:
async def run_warehouse_agent(query: str, session_id: str, session_service: InMemorySessionService) -> str:

    existing_session = await session_service.get_session(
        app_name="warehouse_app",
        user_id="user_1",
        session_id=session_id
    )

    if not existing_session:
        await session_service.create_session(
            app_name="warehouse_app",
            user_id="user_1",
            session_id=session_id
        )

    runner = Runner(
        agent=warehouse_agent,
        session_service=session_service,
        app_name="warehouse_app"
    )

    content = types.Content(
        role="user",
        parts=[
            types.Part(
                text=query
            )
        ]
    )

    final_text = ""
    for event in runner.run(
        user_id="user_1",
        session_id=session_id,
        new_message=content,
        run_config=RunConfig(
            max_llm_calls=5
        )
    ):
        if event.is_final_response():
            if event.content and event.content.parts:
                for part in event.content.parts:
                    final_text += part.text

    return final_text

In [22]:
answer_1 = await run_warehouse_agent(
    query="What is the availability of B09X1LDMH6 in all of your warehouses?",
    session_id="test_01",
    session_service=session_service
)

In [23]:
print(answer_1)

Actions performed:
1. Checked availability of product **B09X1LDMH6** across all warehouses.
2. Result: **No stock available in any warehouse**.

Availability summary:
- **Berlin Distribution Center (DE-BER-01):** 0
- **Lyon Regional Warehouse (FR-LYO-01):** 0
- **Munich Logistics Hub (DE-MUN-01):** 0
- **Paris Central Depot (FR-PAR-01):** 0
- **Marseille Mediterranean Hub (FR-MAR-01):** 0
- **Hamburg North Warehouse (DE-HAM-01):** 0

Order status:
- **Cannot be reserved** because the item is unavailable everywhere.


In [24]:
answer_2 = await run_warehouse_agent(
    query="Can you reserve 9 of this in the Munich warehouse?",
    session_id="test_01",
    session_service=session_service
)

In [25]:
print(answer_2)

Actions performed:
1. Checked availability for **9 units of B09X1LDMH6**.
2. Checked the **Munich warehouse** and all other warehouses for stock.

Result:
- **Munich Logistics Hub (DE-MUN-01): 0 available**
- The item is **not available in any warehouse**.

Reservation status:
- **Cannot reserve 9 units** in Munich.
- **No reservation was made**.
